# M4P2 - Functional Analysis

In this Notebook, we will be performing a functional analysis on the dataset [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275). We used this in the practice M4P1(./M4P1_Intro_to_DE_Analysis.ipynb). We will be using the same interactions as before:
| Coefficient                     | Meaning                | Comparison                                                                  |
|---------------------------------|------------------------|----------------------------------------------------------------------------|
| Intercept                        | Baseline expression    | Col-0 Untreated                                                            |
| genotype_opt3_vs_col0            | Main effect of genotype| opt3 vs Col-0 (Untreated)                                                 |
| treatment_Treated_vs_Untreated   | Main effect of treatment| Treated vs Untreated (Col-0)                                              |
| genotypeopt3.treatmentTreated    | Interaction effect     | (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated)      |

We will cover: 
- Comparing the Wald test to the Likelihood ratio test
- Genomic annotations
- Over-representation analysis
- Functional class scoring / GSEA


```{admonition} Adapted from
:class: tip
**Source:** [Introduction to DGE](https://hbctraining.github.io/Intro-to-DGE/), Harvard Chan Bioinformatics Core (HBC) training team, [hbctraining/Intro-to-DGE](https://github.com/hbctraining/Intro-to-DGE) (functional analysis lessons: over-representation analysis and functional class scoring).

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Changes:** Rewritten for Arabidopsis (TAIR gene IDs, org.At.tair.db/KEGG ath) using the GSE79275 Cd/iron dataset in place of the HBC human Mov10 data; third-party figures were removed.
```

## Setup

We are initializing the same setup as before, except this time it is done in a couple code cells. 

In [ ]:
### Bioconductor and CRAN libraries used
library(DESeq2)
library(tidyverse)
library(RColorBrewer)
library(pheatmap)
library(tximport)
library(ggplot2)
library(ggrepel)
library(biomaRt)

# Import the counts table
DATA_DIR <- Sys.getenv("DATA_DIR", unset = "../../../data")  # repository data/ folder; run data/fetch_data.sh geo first
counts <- read.delim(file.path(DATA_DIR, "geo", "GSE79275", "GSE79275_raw_read_counts.txt"), header = TRUE, stringsAsFactors = FALSE)

# Make rownames the gene IDs (first column)
rownames(counts) <- counts$AGI
counts <- counts[ , -1]  # remove first column after setting rownames

# Ensure all counts are integers
counts <- round(as.matrix(counts))

# Get sample names from counts table
sample_names <- colnames(counts)

# Initialize vectors
genotype <- character(length(sample_names))
tissue <- character(length(sample_names))
treatment <- character(length(sample_names))
bio_rep <- integer(length(sample_names))
tech_rep <- integer(length(sample_names))

# Parse sample names
for (i in seq_along(sample_names)) {
  s <- sample_names[i]
  
  # Genotype
  if (grepl("^C", s)) {
    genotype[i] <- "col0"
  } else if (grepl("^O", s)) {
    genotype[i] <- "opt3"
  }
  
  # Tissue
  if (grepl("R", s)) {
    tissue[i] <- "Root"
  } else if (grepl("S", s)) {
    tissue[i] <- "Shoot"
  }
  
  # Treatment
  if (grepl("T", s)) {
    treatment[i] <- "Treated"
  } else {
    treatment[i] <- "Untreated"
  }
  
  # Biological replicate
  bio_rep[i] <- as.integer(sub(".*?([0-9]+)[-_][0-9]+\\.bam$", "\\1", s))
}

# Assign technical replicates sequentially for each bio_rep
colData <- data.frame(
  sample = sample_names,
  genotype = factor(genotype),
  tissue = factor(tissue),
  treatment = factor(treatment),
  bio_rep = bio_rep,
  stringsAsFactors = FALSE
)

# Order by bio_rep and assign tech_rep
colData <- colData[order(colData$bio_rep, colData$sample), ]
colData$tech_rep <- ave(colData$bio_rep, colData$bio_rep, FUN = seq_along)

# Set row names to sample names for DESeq2
rownames(colData) <- colData$sample
colData$sample <- NULL

# Inspect
colData

# Move to appropriate naming
data <- counts
meta <- colData

# Reorder metadata rows to match the count matrix
meta <- meta[colnames(data), ]
all(colnames(data) %in% rownames(meta))
all(colnames(data) == rownames(meta))

# Add a column identifying unique biological samples
meta$bio_id <- paste(meta$genotype, meta$tissue, meta$treatment, meta$bio_rep, sep="_")

# Sum counts for technical replicates
collapsed_counts <- sapply(unique(meta$bio_id), function(id) {
  rowSums(data[, meta$bio_id == id])
})

# Make it a proper matrix
collapsed_counts <- as.matrix(collapsed_counts)

# Update metadata to include only biological replicates
collapsed_meta <- meta %>%
  distinct(bio_id, .keep_all = TRUE) %>%
  dplyr::select(-tech_rep)
rownames(collapsed_meta) <- collapsed_meta$bio_id

#Now we subset for only roots: 
# Subset metadata for root samples only
root_meta <- collapsed_meta[collapsed_meta$tissue == "Root", ]

# Subset counts matrix to match only the root samples
root_counts <- collapsed_counts[, rownames(root_meta)]

# Optional: check dimensions
dim(root_counts)
dim(root_meta)

#Now we reference our level factors: 
# Ensure Col-0 is the reference for genotype
root_meta$genotype <- relevel(root_meta$genotype, ref = "col0")

# Ensure Untreated is the reference for treatment
root_meta$treatment <- relevel(root_meta$treatment, ref = "Untreated")

In [ ]:
#DESeq2 Object Creation(dds)
dds <- DESeqDataSetFromMatrix(countData = root_counts,
                              colData = root_meta,
                              design = ~ genotype + treatment + genotype:treatment)
#estimate size factors
dds <- estimateSizeFactors(dds)

#normalize counts
normalized_counts <- counts(dds, normalized=TRUE)

### Transform counts for data visualization
vsd <- vst(dds, blind=TRUE)

colnames(colData(dds))

### Extract the rlog matrix from the object
vsd_mat <- assay(vsd)  

### Compute pairwise correlation values
vsd_cor <- cor(vsd_mat)    ## cor() is a base R function

dds <- DESeqDataSetFromMatrix(countData = root_counts,
                              colData = root_meta,
                              design = ~ genotype + treatment + genotype:treatment)

dds <- DESeq(dds)

## Check the size factors
sizeFactors(dds)

## Total number of raw counts per sample
colSums(counts(dds))

## Total number of normalized counts per sample
colSums(counts(dds, normalized=T))

results(dds, name = "genotypeopt3.treatmentTreated")

## Extract results for (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated)
res_tableOE <- results(dds, name = "genotypeopt3.treatmentTreated")

res_tableOE

## Save the unshrunken results to compare
res_tableOE_unshrunken <- res_tableOE

# Apply fold change shrinkage
res_tableOE <- lfcShrink(dds, coef="genotypeopt3.treatmentTreated", type="apeglm")

## Summarize results
summary(res_tableOE, alpha = 0.05)

### Set thresholds
padj.cutoff <- 0.05

# Create a tibble of results
res_tableOE_tb <- res_tableOE %>%
  data.frame() %>%
  rownames_to_column(var="gene") %>% 
  as_tibble()

# Subset the tibble to keep only significant genes
sigOE <- res_tableOE_tb %>%
        dplyr::filter(padj < padj.cutoff)

roots_meta_names <- root_meta %>%
              rownames_to_column(var="samplename") %>% 
              as_tibble()

#Select the genes from sigOE
deg_genes <- sigOE$gene

deg_genes

#Connect to Ensembl Plants for Arabidopsis
mart <- useMart(
  biomart = "plants_mart",
  dataset = "athaliana_eg_gene",
  host = "https://plants.ensembl.org"
)

#Retrieve gene symbols
gene_info <- getBM(
  attributes = c("ensembl_gene_id", "external_gene_name"),
  filters = "ensembl_gene_id",
  values = deg_genes,
  mart = mart
)

#Create a mapping vector
gene_map <- setNames(gene_info$external_gene_name, gene_info$ensembl_gene_id)

#Add a 'common_name' column to sigOE
sigOE <- sigOE %>%
  mutate(common_name = gene_map[gene]) %>%
  mutate(common_name = ifelse(is.na(common_name), gene, common_name))  # keep original ID if no mapping

#Step 6: View the top rows
head(sigOE)

## Exploring results from the Likelihood ratio test (LRT)

DESeq2 also offers the Likelihood Ratio Test as an alternative when evaluating expression change across more than two levels. Genes that are identified as significant are those that are changing in expression in any direction across the different factor levels.

Generally, this test will result in a larger number of genes than the individual pair-wise comparisons. While the LRT is a test of significance for differences of any level(s) of the factor, one should not expect it to be exactly equal to the union of sets of genes using Wald tests (although we do expect a high degree of overlap).

To extract the results from our `dds_lrt` object, we can use the same results() function we had used with the Wald test. There is no need for contrasts since we are not making a pair-wise comparison.

In [ ]:
dds_lrt <- DESeqDataSetFromMatrix(countData = root_counts,
                              colData = root_meta,
                              design = ~ genotype + treatment + genotype:treatment)
#estimate size factors
dds_lrt <- estimateSizeFactors(dds_lrt)

#normalize counts
normalized_counts <- counts(dds_lrt, normalized=TRUE)

dds_lrt <- DESeq(dds_lrt, test="LRT", reduced = ~ 1)


# Extract results for LRT
res_LRT <- results(dds_lrt)

# View results for LRT
res_LRT  


The results table output looks similar to the Wald test results, with identical columns to what we observed previously.
Why are fold changes reported for an LRT test?

For analyses using the likelihood ratio test, the p-values are determined solely by the difference in deviance between the full and reduced model formula. A single log2 fold change is printed in the results table for consistency with other results table outputs, but is not associated with the actual test.

**Columns relevant to the LRT test:**

- baseMean: mean of normalized counts for all samples
- stat: the difference in deviance between the reduced model and the full model
- pvalue: the stat value is compared to a chi-squared distribution to generate a pvalue
- padj: BH adjusted p-values

Additional columns:

- log2FoldChange: log2 fold change
- lfcSE: standard error

#### Identifying significant genes

When filtering significant genes from the LRT we threshold only the `padj` column. How many genes are significant at `padj < 0.05`?

In [ ]:
# Create a tibble for LRT results
res_LRT_tb <- res_LRT %>%
  data.frame() %>%
  rownames_to_column(var="gene") %>% 
  as_tibble()

# Subset to return genes with padj < 0.05
sigLRT_genes <- res_LRT_tb %>% 
  dplyr::filter(padj < padj.cutoff)

# Get number of significant genes
nrow(sigLRT_genes)

# Compare to the numbers we had from Wald test
nrow(sigOE)


### Identifying clusters of genes with shared expression profiles

We now have this list of ~15K significant genes that we know are changing in some way across the three different sample groups. What do we do next?

A good next step is to identify groups of genes that share a pattern of expression change across the sample groups (levels). To do this, we will be using a clustering tool called `degPatterns`from the ‘DEGreport’ package. The `degPatterns` tool uses a hierarchical clustering approach based on pair-wise correlations between genes, then cuts the hierarchical tree to generate groups of genes with similar expression profiles. The tool cuts the tree in a way to optimize the diversity of the clusters, such that the variability inter-cluster > the variability intra-cluster.

Before we begin clustering, we will first subset our rlog-transformed normalized counts to retain only the differentially expressed genes (padj < 0.05). In our case, it may take some time to run the clustering on 7K genes, and so for class demonstration purposes, we will not use the `degPatterns` package. We will use k-means clustering manually.

In [ ]:
### Transform counts for data visualization
rld <- rlog(dds, blind=TRUE)
rld_mat <- assay(rld)

The rlog-transformed counts for the significant genes are turned into clusters with k-means clustering.


In [ ]:
#Use your rlog or vst matrix for significant genes
cluster_rlog <- assay(vsd)[sigOE$gene, ]

# Scale expression values per gene
cluster_scaled <- t(scale(t(cluster_rlog)))

# Run k-means clustering
set.seed(123)
k <- 4  # number of clusters
kmeans_res <- kmeans(cluster_scaled, centers = k)

# Create a cluster dataframe
cluster_df <- data.frame(
  gene = rownames(cluster_scaled),
  cluster = kmeans_res$cluster
)

# Merge scaled expression data with clusters for plotting
plot_df <- cluster_scaled %>%
  as.data.frame() %>%
  rownames_to_column("gene") %>%
  pivot_longer(-gene, names_to = "sample", values_to = "expression") %>%
  left_join(cluster_df, by = "gene")

# Plot average expression per cluster
plot_df %>%
  group_by(cluster, sample) %>%
  summarise(mean_expr = mean(expression), .groups = "drop") %>%
  ggplot(aes(x = sample, y = mean_expr, group = cluster, color = factor(cluster))) +
  geom_line() +
  facet_wrap(~cluster, scales = "free_y") +
  theme_bw() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1)) +
  labs(title = "Expression Clusters (k-means)", y = "Scaled Expression", color = "Cluster")

In [ ]:
# The main gene-to-cluster mapping
clusters$df

In [ ]:
group1 <- clusters$df %>% filter(cluster == 1)
group1

## Genomic annotations

The analysis of next-generation sequencing results requires associating genes, transcripts, proteins, etc. with functional or regulatory information. To perform functional analysis on gene lists, we often need to obtain gene identifiers that are compatible with the tools we wish to use and this is not always trivial. Here, we discuss ways in which you can obtain gene annotation information and some of the advantages and disadvantages of each method.
### Databases

We retrieve information on the processes, pathways, etc. (for which a gene is involved in) from the necessary database where the information is stored. The database you choose will be dependent on what type of information you are trying to obtain. Examples of databases that are often queried, include:

General databases

Offer comprehensive information on genome features, feature coordinates, homology, variant information, phenotypes, protein domain/family information, associated biological processes/pathways, associated microRNAs, etc.:

    Ensembl (use Ensembl gene IDs)
    NCBI (use Entrez gene IDs)
    UCSC
    EMBL-EBI

Annotation-specific databases

Provide annotations related to a specific topic:

    Gene Ontology (GO): database of gene ontology biological processes, cellular components and molecular functions - based on Ensembl or Entrez gene IDs or official gene symbols
    KEGG: database of biological pathways - based on Entrez gene IDs
    MSigDB: database of gene sets
    Reactome: database of biological pathways
    Human Phenotype Ontology: database of genes associated with human disease
    CORUM: database of protein complexes for human, mouse, rat
    
This is by no means an exhaustive list; there are many other databases available that are not listed here.

### Genome builds

Before you begin your search through any of these databases, you should know which build of the genome was used to generate your gene list and make sure you use the same build for the annotations during functional analysis. When a new genome build is acquired, the names and/or coordinate location of genomic features (gene, transcript, exon, etc.) may change. Therefore, the annotations regarding genome features (gene, transcript, exon, etc.) is genome-build specific, and we need to make sure that our annotations are obtained from the appropriate resource.

For example, if we used the TAIR10 build of the A. thaliana to quantify gene expression used for differential expression analysis, then we should use the same TAIR build of the genome to convert between gene IDs and to identify annotations for each of the genes.

### Tools for accessing databases

Within R, there are many popular packages used for gene/transcript-level annotation. These packages provide tools that take the list of genes you provide and retrieve information for each gene using one or more of the databases listed above.
#### Annotation tools: for accessing/querying annotations from a specific databases

| Tool | Description | Pros | Cons |
|------|--------------|------|------|
| **org.Xx.eg.db** | Query gene feature information for the organism of interest | Gene ID conversion, biotype and coordinate information | Only latest genome build available |
| **EnsDb.Xx.vxx** | Transcript and gene-level information directly fetched from Ensembl API (similar to TxDb, but with filtering ability and versioned by Ensembl release) | Easy functions to extract features, direct filtering | Not the most up-to-date annotations, more difficult to use than some packages |
| **TxDb.Xx.UCSC.hgxx.knownGene** | UCSC database for transcript and gene-level information or can create own TxDb from an SQLite database file using the GenomicFeatures package | Feature information, easy functions to extract features | Only available for current and recent genome builds; less up-to-date with annotations than Ensembl |
| **annotables** | Gene-level feature information immediately available for the human and model organisms | Super quick and easy gene ID conversion, biotype and coordinate information | Static resource, not updated regularly |
| **biomaRt** | An R package version of the Ensembl BioMart online tool | All Ensembl database information available, all organisms on Ensembl, wealth of information | — |


#### Interface tools: for accessing/querying annotations from multiple different annotation sources

    AnnotationDbi: queries the OrgDb, TxDb, Go.db, EnsDb, and BioMart annotations.
    AnnotationHub: queries a large collection of whole genome resources, including ENSEMBL, UCSC, ENCODE, Broad Institute, KEGG, NIH Pathway Interaction Database, etc.


#### A. *thaliana* TAIR10 Data Annotation with `AnnotationHub`

### Overview

**AnnotationHub** is a powerful Bioconductor resource for accessing genomic data or querying large collections of whole-genome resources, including **Ensembl**, **UCSC**, **ENCODE**, **Broad Institute**, **KEGG**, and **NIH Pathway Interaction Database**.  
It provides an interface to retrieve and cache data directly from these databases for quick, reproducible access.


In [ ]:
library(AnnotationHub)
library(ensembldb)

# Connect to the AnnotationHub database
ah <- AnnotationHub()

hubCache(ah)   # Location of local cache
hubUrl(ah)     # URL of the online hub


ah

You can explore available species, data object types, and data providers:

In [ ]:
unique(ah$species) %>% View()       # Species
unique(ah$rdataclass) %>% View()    # Data object types
unique(ah$dataprovider) %>% View()  # Data providers

In [ ]:
query(ah, "Arabidopsis")
query(ah, "EnsDb")
query(ah, "TAIR10")

In [ ]:
# Load Arabidopsis OrgDb
ath_orgdb <- ah[["AH114076"]]

# Check keys available
keys(ath_orgdb)[1:10]

# Extract gene-level annotation
annotations <- AnnotationDbi::select(
  ath_orgdb,
  keys = keys(ath_orgdb),
  columns = c("SYMBOL", "GENENAME", "ENTREZID"),
  keytype = "TAIR"
)

head(annotations)

In this section, our focus has been on using annotation packages to extract information mainly just for gene ID conversion, for the different tools that we use downstream. Many of the annotation packages we have presented have much more information than what we need for functional analysis, and we have only just scratched the surface here. It’s good to know the capabilities of the tools we use, so we encourage you to spend some time exploring these packages to become more familiar with them.

## Functional Analysis
The output of RNA-seq differential expression analysis is a list of significantly differentially expressed genes (DEGs). To gain greater biological insight into the differentially expressed genes, there are various analyses that can be done:

- determine whether there is enrichment of known biological functions, interactions, or pathways
- identify genes’ involvement in novel pathways or networks by grouping genes together based on similar trends
- Use global changes in gene expression by visualizing all genes being significantly up- or down-regulated in the context of external interaction data

Generally, for any differential expression analysis, it is useful to interpret the resulting gene lists using freely available web- and R-based tools. While tools for functional analysis span a wide variety of techniques, they can loosely be categorized into three main types: over-representation analysis, functional class scoring, and pathway topology.

*[Pathway-analysis overview figure omitted (third-party figure reproduced in the HBC lessons).]*

The goal of functional analysis is to provide biological insight, so it’s necessary to analyze our results in the context of our experimental hypothesis: Cd has an interference with iron homeostasis and signaling within *A. thaliana*. Therefore, based on the authors’ hypothesis, we may expect the enrichment of processes/pathways related to translation, splicing, and the regulation of mRNAs, which we would need to validate experimentally.

### Over-representation analysis

There are a plethora of functional enrichment tools that perform some type of “over-representation” analysis by querying databases containing information about gene function and interactions.

These databases typically categorize genes into groups (gene sets) based on shared function, or involvement in a pathway, or presence in a specific cellular location, or other categorizations, e.g., functional pathways, etc. Essentially, known genes are binned into categories that have been consistently named (controlled vocabulary) based on how the gene has been annotated functionally. These categories are independent of any organism; however, each organism has distinct categorizations available.

To determine whether any categories are over-represented, you can determine the probability of having the observed proportion of genes associated with a specific category in your gene list based on the proportion of genes associated with the same category in the background set (gene categorizations for the appropriate organism).

![go_proportions.png](images/M4P2_Functional_Analysis_c6f29ad4.png)

The statistical test that will determine whether something is actually over-represented is the Hypergeometric test.

#### Hypergeometric testing

Using the example of the first functional category above, hypergeometric distribution is a probability distribution that describes the probability of 25 genes (k) being associated with “Functional category 1”, for all genes in our gene list (n=1000), from a population of all of the genes in entire genome (N=13,000) which contains 35 genes (K) associated with “Functional category 1” [2].

The calculation of the probability of k successes follows the formula:

![hypergeo.png](images/M4P2_Functional_Analysis_699dc8e2.png)

This test will result in an adjusted p-value (after multiple test correction) for each category tested.

#### Gene Ontology project

One of the most widely used categorizations is the Gene Ontology (GO) established by the Gene Ontology project.

“The Gene Ontology project is a collaborative effort to address the need for consistent descriptions of gene products across databases”. [The Gene Ontology Consortium](http://geneontology.org/page/go-consortium-contributors-list) maintains the GO terms, and these GO terms are incorporated into gene annotations in many of the popular repositories for animal, plant, and microbial genomes.

Tools that investigate enrichment of biological functions or interactions often use the Gene Ontology (GO) categorizations, i.e. the GO terms to determine whether any have significantly modified representation in a given list of genes. Therefore, to best use and interpret the results from these functional analysis tools, it is helpful to have a good understanding of the GO terms themselves and their organization.
GO Ontologies

To describe the roles of genes and gene products, GO terms are organized into three independent controlled vocabularies (ontologies) in a species-independent manner:

- Biological process: refers to the biological role involving the gene or gene product, and could include “transcription”, “signal transduction”, and “apoptosis”. A biological process generally involves a chemical or physical change of the starting material or input.
- Molecular function: represents the biochemical activity of the gene product, such activities could include “ligand”, “GTPase”, and “transporter”.
- Cellular component: refers to the location in the cell of the gene product. Cellular components could include “nucleus”, “lysosome”, and “plasma membrane”.

Each GO term has a term name (e.g. DNA repair) and a unique term accession number (GO:0005125), and a single gene product can be associated with many GO terms, since a single gene product “may function in several processes, contain domains that carry out diverse molecular functions, and participate in multiple alternative interactions with other proteins, organelles or locations in the cell”.

#### GO term hierarchy

Some gene products are well-researched, with vast quantities of data available regarding their biological processes and functions. However, other gene products have very little data available about their roles in the cell.

For example, the protein, “p53”, would contain a wealth of information on its roles in the cell, whereas another protein might only be known as a “membrane-bound protein” with no other information available.

The GO ontologies were developed to describe and query biological knowledge with differing levels of information available. To do this, GO ontologies are loosely hierarchical, ranging from general, ‘parent’, terms to more specific, ‘child’ terms. The GO ontologies are “loosely” hierarchical since ‘child’ terms can have multiple ‘parent’ terms.

Some genes with less information may only be associated with general ‘parent’ terms or no terms at all, while other genes with a lot of information be associated with many terms.

*[GO hierarchy figure omitted (third-party journal figure reproduced in the HBC lessons).]*

[Tips for working with GO terms](http://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1003343)


#### clusterProfiler

We will be using `clusterProfiler` to perform over-representation analysis on GO terms associated with our list of significant genes. The tool takes as input a significant gene list and a background gene list and performs statistical enrichment analysis using hypergeometric testing. The basic arguments allow the user to select the appropriate organism and GO ontology (BP, CC, MF) to test.

**Running clusterProfiler**

To run clusterProfiler GO over-representation analysis, we will change our gene names into Ensembl IDs, since the tool works a bit easier with the Ensembl IDs.

Then load the following libraries:

In [ ]:
# Load libraries
library(DOSE)
library(pathview)
library(clusterProfiler)
library(org.At.tair.db)

For the different steps in the functional analysis, we require Ensembl and Entrez IDs. We will use the gene annotations that we generated previously to merge with our differential expression results. Before we do that, let’s subset our results tibble to only have the genes that were tested, i.e. genes whose p-adjusted values are not equal to “NA”.

In [ ]:
## Untested genes have padj = NA, so let's keep genes with padj != NA
res_tableOE_tb_noNAs <- filter(res_tableOE_tb, padj != "NA" )
res_tableOE_tb_noNAs
annotations
## Merge the AnnotationHub dataframe with the results 
res_ids <- left_join(res_tableOE_tb_noNAs, annotations, by=c("gene"="TAIR")) 

res_ids_unique <- res_ids %>%
  group_by(gene) %>%
  slice(1) %>%      # keep first row per TAIR
  ungroup()

To perform the over-representation analysis, we need a list of background genes and a list of significant genes. For our background dataset we will use all genes tested for differential expression (all genes in our results table). For our significant gene list we will use genes with p-adjusted values less than 0.05 (we could include a fold change threshold too if we have many DE genes).

In [ ]:
## Create background dataset for hypergeometric testing using all tested genes for significance in the results                 
allOE_genes <- as.character(res_ids$gene)

## Extract significant results
sigOE <- dplyr::filter(res_ids, padj < 0.05)

sigOE_genes <- as.character(sigOE$gene)

Now we can perform the GO enrichment analysis and save the results:

In [ ]:
## Run GO enrichment analysis 
ego <- enrichGO(
  gene = sigOE_genes,
  universe = allOE_genes,
  keyType = "TAIR",        # TAIR IDs for Arabidopsis
  OrgDb = org.At.tair.db,
  ont = "BP",              # Biological Process
  pAdjustMethod = "BH",
  qvalueCutoff = 0.05,
  readable = TRUE
)

# View results
head(ego)

In [ ]:
## Output results from GO analysis to a table
cluster_summary <- data.frame(ego)

dir.create("results", showWarnings = FALSE)
write.csv(cluster_summary, "results/clusterProfiler_GSE79275.csv")

#### Visualizing clusterProfiler results

`clusterProfiler` has a variety of options for viewing the over-represented GO terms. We will explore the dotplot, enrichment plot, and the category netplot.

The dotplot shows the number of genes associated with the first 50 terms (size) and the p-adjusted values for these terms (color). This plot displays the top 50 GO terms by gene ratio (# genes related to GO term / total number of sig genes), not p-adjusted value.

In [ ]:
dotplot(ego, showCategory=10)

The next plot is the enrichment GO plot, which shows the relationship between the top 50 most significantly enriched GO terms (padj.), by grouping similar terms together. Before creating the plot, we will need to obtain the similarity between terms using the pairwise_termsim() function (instructions for emapplot). In the enrichment plot, the color represents the p-values relative to the other displayed terms (brighter red is more significant), and the size of the terms represents the number of genes that are significant from our list.

In [ ]:
## Add similarity matrix to the termsim slot of enrichment result
ego <- enrichplot::pairwise_termsim(ego)

## Enrichmap clusters the 15 most significant (by padj) GO terms to visualize relationships between terms
emapplot(ego, showCategory = 20)

Finally, the category netplot shows the relationships between the genes associated with the top five most significant GO terms and the fold changes of the significant genes associated with these terms (color). The size of the GO terms reflects the number of genes in the terms, with terms with more genes being larger. This plot is particularly useful for hypothesis generation in identifying genes that may be important to several of the most affected processes.

In [ ]:
## To color genes by log2 fold changes, we need to extract the log2 fold changes from our results table creating a named vector
OE_foldchanges <- sigOE$log2FoldChange

names(OE_foldchanges) <- sigOE$gene

## Cnetplot details the genes associated with one or more terms - by default gives the top 5 significant terms (by padj)
cnetplot(ego, 
         showCategory = 5, 
         color.params=list(foldChange=OE_foldchanges),
         vertex.label.font=6)
         
## If some of the high fold changes are getting drowned out due to a large range, you could set a maximum fold change value
OE_foldchanges <- ifelse(OE_foldchanges > 2, 2, OE_foldchanges)
OE_foldchanges <- ifelse(OE_foldchanges < -2, -2, OE_foldchanges)

cnetplot(ego, 
         showCategory = 5, 
         color.params=list(foldChange=OE_foldchanges),
         vertex.label.font=6)

If you are interested in significant processes that are not among the top five, you can subset your ego dataset to only display these processes:

In [ ]:
## Subsetting the ego results without overwriting original `ego` variable
ego2 <- ego

ego2@result <- ego@result[c(1,3,4,8,9),]

## Plotting terms of interest
cnetplot(ego2, 
         categorySize="pvalue", 
         foldChange=OE_foldchanges, 
         showCategory = 5, 
         vertex.label.font=4)

## Functional analysis using functional class scoring

In addition to over-representation analysis, there are other types of analyses can be equally important or informative for obtaining some biological insight from your results. The hypothesis behind functional class scoring (FCS) methods is that although large changes in individual genes can have significant effects on pathways (and will be detected via ORA methods), weaker but coordinated changes in sets of functionally related genes (i.e., pathways) can also have significant effects. Thus, rather than setting an arbitrary threshold to identify ‘significant genes’, all genes are considered in the analysis. The gene-level statistics from the dataset are aggregated to generate a single pathway-level statistic and statistical significance of each pathway is reported. This type of analysis can be particularly helpful if the differential expression analysis only outputs a small list of significant DE genes.

*[Pathway-analysis overview figure omitted (third-party figure reproduced in the HBC lessons).]*

### Gene set enrichment analysis using clusterProfiler and Pathview

One commonly used tool which is classified under Functional class scoring (FCS), is GSEA. Gene set enrichment analysis utilizes the gene-level statistics or log2 fold changes for all genes to look to see whether gene sets for particular biological pathways are enriched among the large positive or negative fold changes.

![gsea_theory.png](images/M4P2_Functional_Analysis_4baed27f.png)

*Image: Harvard Chan Bioinformatics Core (HBC), CC BY 4.0.*

Gene sets are pre-defined groups of genes, which are functionally related. Commonly used gene sets include those derived from KEGG pathways, Gene Ontology terms, MSigDB, Reactome, or gene groups that share some other functional annotations, etc. [1].

#### Preparation for GSEA

The clusterProfiler package offers several functions to perform GSEA using different genes sets, including but not limited to GO, KEGG, and MSigDb. We will use the KEGG gene sets in our examples below. The KEGG gene sets are defined using the Entrez identifiers, thus to perform the analysis we will need to acquire the corresponding Entrez IDs for our genes. We will also need to remove any genes that do not have an Entrez ID (NA values) and any duplicates (due to gene ID conversion) that may exist:

In [ ]:
# Remove rows with NA Entrez IDs
res_entrez <- res_ids %>%
  filter(!is.na(ENTREZID))

# Remove duplicate Entrez IDs, keep first occurrence
res_entrez <- res_entrez[!duplicated(res_entrez$ENTREZID), ]

GSEA will use the log2 fold changes obtained from the differential expression analysis for every gene, to perform the analysis. We will obtain a vector of fold changes for input to clusterProfiler, in addition to the associated Entrez IDs:

In [ ]:
## Extract the foldchanges
foldchanges <- res_entrez$log2FoldChange

## Name each fold change with the corresponding Entrez ID
names(foldchanges) <- res_entrez$ENTREZID

Next we need to order the fold changes in decreasing order. To do this we’ll use the sort() function, which takes a vector as input. This is in contrast to Tidyverse’s arrange(), which requires a data frame.

In [ ]:
## Sort fold changes in decreasing order
foldchanges <- sort(foldchanges, decreasing = TRUE)

head(foldchanges)

### Theory of GSEA

Now we are ready to perform GSEA. The details regarding GSEA can be found in the PNAS paper by Subramanian et al. We will describe briefly the steps outlined in the paper below:

*[Figure omitted: Subramanian et al. (2005) PNAS 102:15545-15550, Fig. 1, [doi:10.1073/pnas.0506580102](https://doi.org/10.1073/pnas.0506580102).]*

Image credit: Subramanian et al. Proceedings of the National Academy of Sciences Oct 2005, 102 (43) 15545-15550; DOI: 10.1073/pnas.0506580102

This image describes the theory of GSEA, with the ‘gene set S’ showing the metric used (in our case, ranked log2 fold changes) to determine enrichment of genes in the gene set. The left-most image is representing this metric used for the GSEA analysis. The log2 fold changes for each gene in the ‘gene set S’ is shown as a line in the middle image. The large positive log2 fold changes are at the top of the gene set image, while the largest negative log2 fold changes are at the bottom of the gene set image. In the right-most image, the gene set is turned horizontally, underneath which is an image depicting the calculations involved in determining enrichment, as described below.

**Step 1:** Calculation of enrichment score:

An enrichment score for a particular gene set is calculated by walking down the list of log2 fold changes and increasing the running-sum statistic every time a gene in the gene set is encountered and decreasing it when genes are not part of the gene set. The size of the increase/decrease is determined by magnitude of the log2 fold change. Larger (positive or negative) log2 fold changes will result in larger increases or decreases. The final enrichment score is where the running-sum statistic is the largest deviation from zero.

**Step 2:** Estimation of significance:

The significance of the enrichment score is determined using permutation testing, which performs rearrangements of the data points to determine the likelihood of generating an enrichment score as large as the enrichment score calculated from the observed data. Essentially, for this step, the first permutation would reorder the log2 fold changes and randomly assign them to different genes, reorder the gene ranks based on these new log2 fold changes, and recalculate the enrichment score. The second permutation would reorder the log2 fold changes again and recalculate the enrichment score again, and this would continue for the total number of permutations run. Therefore, the number of permutations run will increase the confidence in the signficance estimates.

**Step 3:** Adjust for multiple test correction

After all gene sets are tested, the enrichment scores are normalized for the size of the gene set, then the p-values are corrected for multiple testing.

The GSEA output will yield the core genes in the gene sets that most highly contribute to the enrichment score. The genes output are generally the genes at or before the running sum reaches its maximum value (eg. the most influential genes driving the differences between conditions for that gene set).

#### Performing GSEA

First, we will set the seed so that we all obtain the same result, then to perform the GSEA using KEGG gene sets with clusterProfiler, we can use the gseKEGG() function. The organisms with available KEGG pathway information are listed here.

In [ ]:
# Keep only genes with valid Entrez IDs
res_entrez <- res_ids %>%
  filter(!is.na(ENTREZID)) %>%
  distinct(ENTREZID, .keep_all = TRUE)

# Map Entrez IDs to KEGG IDs
entrez2kegg <- bitr_kegg(res_entrez$ENTREZID,
                         fromType = "ncbi-geneid",
                         toType = "kegg",
                         organism = "ath")

# Merge KEGG IDs back with fold changes
res_kegg <- res_entrez %>%
  inner_join(entrez2kegg, by = c("ENTREZID" = "ncbi-geneid"))

# Named fold change vector
foldchanges <- res_kegg$log2FoldChange
names(foldchanges) <- res_kegg$kegg

# Sort decreasing
foldchanges <- sort(foldchanges, decreasing = TRUE)

# Run GSEA
set.seed(123456)
gseaKEGG <- gseKEGG(geneList = foldchanges,
                    organism = "ath",
                    minGSSize = 20,
                    pvalueCutoff = 0.05,
                    verbose = FALSE)

# Results
gseaKEGG_results <- gseaKEGG@result
dir.create("results", showWarnings = FALSE)
write.csv(gseaKEGG_results, "results/gsea_kegg_GSE79275.csv", quote=F)
head(gseaKEGG_results)


- The first few columns of the results table identify the pathway information
- The following columns include the associated statistics
- The last column will report which genes are part of the ‘core enrichment’. These are the genes associated with the pathway which contributed to the observed enrichment score (i.e. in the extremes of the ranking). The genes are listed by EntrezID.

Let’s explore the GSEA plot of enrichment of one of the pathways in the ranked list:

In [ ]:
## Plot the GSEA plot for a single enriched pathway, `ath03430`
gseaplot(gseaKEGG, geneSetID = 'ath03430')

In this plot, the lines in plot represent the genes in the gene set ‘ath03430’, and where they occur among the log2 fold changes. The largest positive log2 fold changes are on the left-hand side of the plot, while the largest negative log2 fold changes are on the right. The top plot shows the magnitude of the log2 fold changes for each gene, while the bottom plot shows the running sum, with the enrichment score peaking at the red dotted line (which is among the negative log2 fold changes). This suggests the down-regulation of this pathway.

Use the [Pathview R package](http://bioconductor.org/packages/release/bioc/html/pathview.html) to integrate the KEGG pathway data from clusterProfiler.

In [ ]:
# May or may not work depending on the gene used.
# Load libraries
library(dplyr)
library(clusterProfiler)
library(pathview)

pathview(gene.data = foldchanges,
         pathway.id = "ath04814",
         species = "ath",
         limit = list(gene = 2, cpd = 1))


### Other Tools for Functional Analysis
Co-expression clustering

Co-expression clustering is often used to identify genes of novel pathways or networks by grouping genes together based on similar trends in expression. These tools are useful in identifying genes in a pathway, when their participation in a pathway and/or the pathway itself is unknown. These tools cluster genes with similar expression patterns to create ‘modules’ of co-expressed genes which often reflect functionally similar groups of genes. These ‘modules’ can then be compared across conditions or in a time-course experiment to identify any biologically relevant pathway or network information.

You can visualize co-expression clustering using heatmaps, which should be viewed as suggestive only; serious classification of genes needs better methods.

The way the tools perform clustering is by taking the entire expression matrix and computing pair-wise co-expression values. A network is then generated from which we explore the topology to make inferences on gene co-regulation. The WGCNA package (in R) is one example of a more sophisticated method for co-expression clustering (please note however, that it is no longer maintained).

#### Resources for functional analysis

- [g:Profiler](http://biit.cs.ut.ee/gprofiler/index.cgi)
- [DAVID](https://david.ncifcrf.gov)
- [clusterProfiler](http://bioconductor.org/packages/release/bioc/html/clusterProfiler.html)
- [GeneMANIA](http://www.genemania.org/)
- [GenePattern](http://www.broadinstitute.org/cancer/software/genepattern/) (need to register)
- [WebGestalt](http://www.webgestalt.org) (need to register)
- [AmiGO](http://amigo.geneontology.org/amigo)
- [ReviGO (visualizing GO analysis, input is GO terms)](http://revigo.irb.hr)
- [WGCNA](https://horvath.genetics.ucla.edu/html/CoexpressionNetwork/Rpackages/WGCNA/) (no longer maintained)
- [GSEA](http://software.broadinstitute.org/gsea/index.jsp)
- [SPIA](https://www.bioconductor.org/packages/release/bioc/html/SPIA.html)
- [GAGE/Pathview](http://www.bioconductor.org/packages/release/bioc/html/gage.html)


*Adapted from [hbctraining - Intro-to-DGE](https://hbctraining.github.io/Intro-to-DGE/schedule/links-to-lessons.html)*